# 06 — Allie embeddings + 3 tabular features

Train a single move-level MLP on ChessFraud-Synth using:
- **Allie** embedding (1024-D)
- **3 tabular features**: `sf15_match`, `player_elo_z`, `opponent_elo_z`

Total input dimension: **1027**.

Cheat sources (per-row uniform mixture, 1 cheat sample per human sample):
`stockfish_1, stockfish_9, stockfish_15, lc0_1, lc0_100`.

Training on **synth train**; evaluation on **synth test** (same pair construction)
and on **tournament** (no pair construction — each row has one labeled move).

Main metric: **Macro F1**. Reported alongside cheat-class Precision / Recall / F1.

All artefacts (logs, metrics, plots, **per-sample predictions in parquet**)
are saved under `reports/move_level/06_allie_plus_4features/`. The parquet files are
the input to the case-study (step 2).

In [ ]:
# Cell 1. Imports, seeds, device, run_dir.
import json
import logging
import os
import platform
import socket
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

_REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from experiments.move_level.configs.dataset_config import DatasetConfig
from experiments.move_level.configs.model_config import ModelConfig, TrainConfig
from experiments.move_level.features.feature_config import FeatureConfig
from experiments.move_level.features.feature_builder import enabled_feature_names
from experiments.move_level.data.synth_dataset import load_synth_data, build_dataloaders
from experiments.move_level.data.tournament_dataset import load_tournament_data
from experiments.move_level.training.trainer import train_one
from experiments.move_level.evaluation.synth_eval import evaluate_on_synth_test
from experiments.move_level.evaluation.tournament_eval import (
    evaluate_on_tournament, save_eval_results, metrics_to_long_df,
)
from experiments.move_level.evaluation.plots import plot_all_heatmaps
from experiments.move_level.utils.paths import (
    synth_csv_path, synth_emb_allie_path,
    tournament_csv_path, tournament_emb_allie_path,
    ensure_out_dir,
)
from experiments.move_level.utils.device import get_device
from experiments.move_level.utils.random import set_all_seeds

SEED = 42
set_all_seeds(SEED)
device = get_device()

RUN_TAG = "06_allie_plus_4features"
run_dir = _REPO_ROOT / "reports" / "move_level" / RUN_TAG
ensure_out_dir(run_dir)
(run_dir / "predictions").mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 1b. Logging setup (file + stdout) and header.
log = logging.getLogger("move_level")
log.setLevel(logging.DEBUG)
for h in list(log.handlers):
    log.removeHandler(h)
_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
_fh = logging.FileHandler(run_dir / "train.log", mode="w", encoding="utf-8")
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt); log.addHandler(_fh)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO); _sh.setFormatter(_fmt); log.addHandler(_sh)
logging.getLogger("move_level.train").handlers = log.handlers
logging.getLogger("move_level.train").setLevel(logging.DEBUG)

def _git_rev():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "?"

log.info("=" * 78)
log.info("RUN TAG    : %s", RUN_TAG)
log.info("started at : %s", datetime.now().isoformat(timespec="seconds"))
log.info("hostname   : %s", socket.gethostname())
log.info("platform   : %s", platform.platform())
log.info("python / torch : %s / %s", sys.version.split()[0], torch.__version__)
log.info("device     : %s", device)
log.info("git rev    : %s  |  run dir: %s", _git_rev(), run_dir)
log.info("=" * 78)

In [ ]:
# ------------------------------------------------------------
# Cell 2. Configs.
# ------------------------------------------------------------

CHEAT_SOURCES = ["stockfish_1", "stockfish_9", "stockfish_15", "lc0_1", "lc0_100"]

# Every cheat source is allowed on every rating bin in this experiment.
N_RATING_BINS = 6
cheat_model_to_bin_idxs = {name: list(range(N_RATING_BINS)) for name in CHEAT_SOURCES}

# --- Paths. Override via environment variables if data lives elsewhere. ---
#   MOVE_LEVEL_SYNTH_CSV, MOVE_LEVEL_SYNTH_NPZ,
#   MOVE_LEVEL_TOURN_CSV, MOVE_LEVEL_TOURN_NPZ.
p_synth_csv = os.environ.get("MOVE_LEVEL_SYNTH_CSV", str(synth_csv_path(_REPO_ROOT)))
p_synth_npz = os.environ.get("MOVE_LEVEL_SYNTH_NPZ", str(synth_emb_allie_path(_REPO_ROOT)))
p_tourn_csv = os.environ.get("MOVE_LEVEL_TOURN_CSV", str(tournament_csv_path(_REPO_ROOT)))
p_tourn_npz = os.environ.get("MOVE_LEVEL_TOURN_NPZ", str(tournament_emb_allie_path(_REPO_ROOT)))
for p, label in [(p_synth_csv, "synth_csv"), (p_synth_npz, "synth_npz"),
                 (p_tourn_csv, "tourn_csv"), (p_tourn_npz, "tourn_npz")]:
    marker = "OK " if Path(p).exists() else "MISS"
    log.info("path %s %-10s %s", marker, label, p)

ds_cfg = DatasetConfig(
    synth_csv_path=p_synth_csv,
    synth_emb_npz_path=p_synth_npz,
    tournament_csv_path=p_tourn_csv,
    tournament_emb_npz_path=p_tourn_npz,
    cheat_models=CHEAT_SOURCES,
    cheat_model_to_bin_idxs=cheat_model_to_bin_idxs,
    build_all_mixture=True,
    batch_size=1024,
    num_workers=0,
    seed=SEED,
    emb_key_human="move_uci",
)

feat_cfg = FeatureConfig(
    sf15_match=True,
    player_elo=True,
    opponent_elo=True,
    standardize=True,
)

# mlp_002 defaults (same as notebook 04):
#   hidden_dim=256, GELU, BatchNorm, dropout=0.0, lr=3e-4, wd=3e-4, epochs=500,
#   early_stop_patience=20, scheduler_patience=10, scheduler_factor=0.1.
model_cfg = ModelConfig()
train_cfg = TrainConfig()

log.info("DatasetConfig:  %s", ds_cfg)
log.info("FeatureConfig:  %s", feat_cfg)
log.info("ModelConfig:    %s", model_cfg)
log.info("TrainConfig:    %s", train_cfg)

In [ ]:
# ------------------------------------------------------------
# Cell 3. Load synth data; log diagnostics; persist feature_stats.
# ------------------------------------------------------------

log.info("Loading synth data from %s", ds_cfg.synth_csv_path)
synth_data = load_synth_data(ds_cfg, feat_cfg)

log.info("synth: N_filtered=%d  emb_dim=%d  input_dim=%d",
         synth_data.vec_human.shape[0], synth_data.emb_human.shape[1],
         synth_data.input_dim)
log.info("synth split sizes: train=%d  val=%d  test=%d",
         len(synth_data.train_idx), len(synth_data.val_idx), len(synth_data.test_idx))
log.info("cheat sources in vec_cheat: %s", list(synth_data.vec_cheat.keys()))
log.info("feature_stats: %s", synth_data.feature_stats)

# --- sanity checks: shapes & sf15_match distribution ---
assert synth_data.vec_human.shape[1] == 1024 + 3, (
    f"unexpected input_dim {synth_data.vec_human.shape[1]}; expected 1027"
)
enabled = enabled_feature_names(feat_cfg)
log.info("enabled features (in concat order): %s", enabled)
sf15_offset = 1024 + enabled.index("sf15_match")
sf15_cheat15 = synth_data.vec_cheat["stockfish_15"][:, sf15_offset]
assert np.allclose(sf15_cheat15, 1.0), (
    "sanity check failed: cheat=stockfish_15 rows must have sf15_match == 1"
)
# human sf15_match rate per rating bin (diagnostic)
sf15_human = synth_data.vec_human[:, sf15_offset]
for b_i, b_name in enumerate(synth_data.bin_names):
    mask = synth_data.bin_idx == b_i
    if mask.any():
        log.info("  human sf15_match rate [%s]: %.4f (N=%d)",
                 b_name, float(sf15_human[mask].mean()), int(mask.sum()))
# per-cheat sf15_match rate
for name in synth_data.cheat_names:
    arr = synth_data.vec_cheat[name][:, sf15_offset]
    log.info("  cheat sf15_match rate [%s]: %.4f", name, float(arr.mean()))

# persist feature_stats.json
(run_dir / "feature_stats.json").write_text(json.dumps(synth_data.feature_stats, indent=2))
log.info("feature_stats saved -> %s", run_dir / "feature_stats.json")

In [ ]:
# ------------------------------------------------------------
# Cell 4. Build dataloaders for cheat_name='ALL' and train.
# ------------------------------------------------------------

CHEAT_TAG = "ALL"
MLP_TAG = "mlp_002"
model_out_dir = run_dir / "models" / CHEAT_TAG / MLP_TAG

train_loader, val_loader, test_loader, y_train, y_val, y_test = build_dataloaders(
    synth_data,
    cheat_name=CHEAT_TAG,
    batch_size=ds_cfg.batch_size,
    num_workers=ds_cfg.num_workers,
)
log.info("dataloaders: train=%d batches  val=%d  test=%d  batch_size=%d",
         len(train_loader), len(val_loader), len(test_loader), ds_cfg.batch_size)

train_result = train_one(
    train_loader=train_loader,
    val_loader=val_loader,
    y_val=y_val,
    input_dim=synth_data.input_dim,
    model_config=model_cfg,
    train_config=train_cfg,
    device=device,
    out_dir=model_out_dir,
    tag=f"{CHEAT_TAG}/{MLP_TAG}",
    logger=logging.getLogger("move_level.train"),
    log_every=1,
)

# Augment meta.json with feature_stats + feature_config for fully reproducible inference.
meta = json.loads(train_result.meta_path.read_text())
meta["feature_stats"] = synth_data.feature_stats
meta["feature_config"] = feat_cfg.__dict__
meta["enabled_features"] = enabled
meta["cheat_sources"] = CHEAT_SOURCES
train_result.meta_path.write_text(json.dumps(meta, indent=2))
log.info("model saved -> %s", train_result.model_path)
log.info("meta  saved -> %s", train_result.meta_path)

In [ ]:
# ------------------------------------------------------------
# Cell 5. Evaluate on synth test (cheat_name='ALL') per rating bin + ALL.
#         Dump per-sample predictions to parquet.
# ------------------------------------------------------------

synth_results, synth_preds = evaluate_on_synth_test(
    synth_data=synth_data,
    cheat_name=CHEAT_TAG,
    model_path=train_result.model_path,
    model_config=model_cfg,
    threshold=train_result.best_thr,
    input_dim=synth_data.input_dim,
    device=device,
    batch_size=4096,
    out_dir=run_dir,
    return_predictions=True,
)

synth_metrics_df = metrics_to_long_df(synth_results, index_name="bin")
synth_metrics_df.to_csv(run_dir / "tables" / "synth_metrics.csv", index=False)
log.info("synth metrics per bin:\n%s", synth_metrics_df.to_string(index=False))

synth_preds_path = run_dir / "predictions" / "synth_test.parquet"
synth_preds.to_parquet(synth_preds_path, index=False)
log.info("synth predictions saved -> %s  (shape=%s)", synth_preds_path, synth_preds.shape)
log.info("synth predictions dtypes:\n%s", synth_preds.dtypes.to_string())

In [ ]:
# ------------------------------------------------------------
# Cell 6. Load tournament + evaluate per Elo bin + ALL.
#         Dump per-sample predictions to parquet.
# ------------------------------------------------------------

log.info("Loading tournament data from %s", ds_cfg.tournament_csv_path)
tournament_data = load_tournament_data(
    ds_cfg, feat_cfg, feature_stats=synth_data.feature_stats,
)
log.info("tournament: N_filtered=%d  input_dim=%d  bins=%s",
         tournament_data.X.shape[0], tournament_data.input_dim, tournament_data.bin_names)
log.info("tournament label distribution: %s",
         pd.Series(tournament_data.y).value_counts().to_dict())

tourn_results, tourn_preds = evaluate_on_tournament(
    tournament_data=tournament_data,
    model_path=train_result.model_path,
    model_config=model_cfg,
    threshold=train_result.best_thr,
    input_dim=tournament_data.input_dim,
    device=device,
    batch_size=4096,
    out_dir=run_dir,
    cheat_name=CHEAT_TAG,
    return_predictions=True,
)

tourn_metrics_df = metrics_to_long_df(tourn_results, index_name="bin")
tourn_metrics_df.to_csv(run_dir / "tables" / "tournament_metrics.csv", index=False)
log.info("tournament metrics per bin:\n%s", tourn_metrics_df.to_string(index=False))

tourn_preds_path = run_dir / "predictions" / "tournament.parquet"
tourn_preds.to_parquet(tourn_preds_path, index=False)
log.info("tournament predictions saved -> %s  (shape=%s)", tourn_preds_path, tourn_preds.shape)
log.info("tournament predictions dtypes:\n%s", tourn_preds.dtypes.to_string())

In [ ]:
# ------------------------------------------------------------
# Cell 7. Heatmaps + final summary.
# ------------------------------------------------------------

all_results_synth = {CHEAT_TAG: synth_results}
all_results_tourn = {CHEAT_TAG: tourn_results}

synth_tables = save_eval_results(
    all_results_synth,
    out_dir=run_dir / "synth_heatmap_tables",
    row_names=list(synth_data.bin_names) + ["ALL"],
)
tourn_tables = save_eval_results(
    all_results_tourn,
    out_dir=run_dir / "tournament_heatmap_tables",
    row_names=list(tournament_data.bin_names),
)
plot_all_heatmaps(synth_tables, run_dir / "figures" / "synth")
plot_all_heatmaps(tourn_tables, run_dir / "figures" / "tournament")

def _fmt(v):
    try:
        return f"{float(v):.4f}"
    except Exception:
        return str(v)

log.info("=" * 78)
log.info("SUMMARY")
log.info("synth   [ALL/ALL]  macro_f1=%s  cheat_p=%s  cheat_r=%s  cheat_f1=%s",
         _fmt(synth_results["ALL"]["macro_f1"]),
         _fmt(synth_results["ALL"]["cheat_precision"]),
         _fmt(synth_results["ALL"]["cheat_recall"]),
         _fmt(synth_results["ALL"]["cheat_f1"]))
log.info("tournament [ALL]   macro_f1=%s  cheat_p=%s  cheat_r=%s  cheat_f1=%s",
         _fmt(tourn_results["ALL"]["macro_f1"]),
         _fmt(tourn_results["ALL"]["cheat_precision"]),
         _fmt(tourn_results["ALL"]["cheat_recall"]),
         _fmt(tourn_results["ALL"]["cheat_f1"]))
log.info("artefacts in %s:", run_dir)
for sub in ["models/ALL/mlp_002/model.pt",
            "models/ALL/mlp_002/meta.json",
            "models/ALL/mlp_002/curves.csv",
            "feature_stats.json",
            "train.log",
            "tables/synth_metrics.csv",
            "tables/tournament_metrics.csv",
            "predictions/synth_test.parquet",
            "predictions/tournament.parquet"]:
    p = run_dir / sub
    log.info("  %s  %s", "OK " if p.exists() else "MISS", p)
log.info("=" * 78)
log.info("DONE at %s", datetime.now().isoformat(timespec="seconds"))